In [1]:
from_date = 20250228
to_date = from_date

from datetime import datetime
# Convert dates to UNIX timestamps
from_date_unix = int(datetime.strptime(str(from_date) + ' 00:00:00', '%Y%m%d %H:%M:%S').timestamp()) + 25200 # Add 7 hours
to_date_unix = int(datetime.strptime(str(to_date) + ' 23:59:59', '%Y%m%d %H:%M:%S').timestamp()) + 25200 # Add 7 hours

print(from_date_unix)
print(to_date_unix)

1740700800
1740787199


In [9]:
from helper.config import load_config
import psycopg2

import pandas as pd
import numpy as np

config = load_config()
with psycopg2.connect(**config) as conn:
    with conn.cursor() as cur:
            cur.execute(f"""
                    select 
                        a.start_time
                        ,a.end_time
                        ,a.start_location_id
                        ,a.end_location_id
                        ,c.lat as start_lat
                        ,c.lon as start_lon
                        ,d.lat as end_lat
                        ,d.lon as end_lon
                        ,a.vehicle_type
                        ,a.distance_meters
                        ,c.location_name as start_location_name
                        ,c.category as start_category
                        ,d.location_name as end_location_name
                        ,d.category as end_category
                        ,case 
                            when json_agg(
                                json_build_object('txtime', b.txtime, 'lat', b.lat, 'lon', b.lon)
                                order by b.txtime
                            ) filter (where b.txtime is not null or b.lat is not null or b.lon is not null) is null 
                            then null
                            else json_agg(
                                json_build_object('txtime', b.txtime, 'lat', b.lat, 'lon', b.lon)
                                order by b.txtime
                            )
                        end as path 
                    from gmap.fact_activity a
                    left join gmap.fact_timelinepath b
                    on b.txtime between a.start_time and a.end_time 
                    left join gmap.dim_location c
                    on a.start_location_id = c.location_id
                    left join gmap.dim_location d
                    on a.end_location_id = d.location_id
                    where a.start_time between {from_date_unix} AND {to_date_unix}
                    group by 
                        a.start_time
                        ,a.end_time
                        ,a.start_location_id
                        ,a.end_location_id
                        ,c.lat
                        ,c.lon
                        ,d.lat
                        ,d.lon
                        ,a.vehicle_type
                        ,a.distance_meters
                        ,c.location_name
                        ,c.category
                        ,d.location_name
                        ,d.category
                        """)
            activity = pd.DataFrame(cur.fetchall(), columns=[desc[0] for desc in cur.description]) 

            cur.execute(f"""
                    select 
                        a.start_time 
                        ,a.end_time 
                        ,a.location_id
                        ,c.lat 
                        ,c.lon
                        ,c.location_name
                        ,c.category
                        ,case 
                            when json_agg(
                                json_build_object('txtime', b.txtime, 'lat', b.lat, 'lon', b.lon)
                                order by b.txtime
                            ) filter (where b.txtime is not null or b.lat is not null or b.lon is not null) is null 
                            then null
                            else json_agg(
                                json_build_object('txtime', b.txtime, 'lat', b.lat, 'lon', b.lon)
                                order by b.txtime
                            )
                        end as path
                    from gmap.fact_visit a
                    left join gmap.fact_timelinepath b
                    on b.txtime between a.start_time and a.end_time 
                    left join gmap.dim_location c
                    on a.location_id = c.location_id
                    where a.start_time between {from_date_unix} AND {to_date_unix}
                    group by 
                        a.start_time 
                        ,a.end_time 
                        ,a.location_id
                        ,c.lat 
                        ,c.lon
                        ,c.location_name
                        ,c.category
                        """)
            visit = pd.DataFrame(cur.fetchall(), columns=[desc[0] for desc in cur.description]) 

# add info to activity
activity['path'] = activity['path'].apply(lambda x: {item['txtime']: [item['lat'], item['lon']] for item in x} if x is not None else None)
activity['color'] = activity['vehicle_type'].map({
    'in bus': 'red',
    'in passenger vehicle': 'blue',
    'walking': 'green',
    'motorcycling': 'magenta',
    'in train': 'yellow',
    'in subway': 'cyan',
    'unknown': 'black'
})
activity['start_time_human'] = pd.to_datetime(activity['start_time'], unit='s')
activity['end_time_human'] = pd.to_datetime(activity['end_time'], unit='s')

# add info to visit
visit['path'] = visit['path'].apply(lambda x: {item['txtime']: [item['lat'], item['lon']] for item in x} if x is not None else None)
visit['duration'] = visit['end_time'] - visit['start_time']
visit['duration'] = visit['duration'].apply(lambda x: '{} hours {} minutes'.format(int(divmod(x, 60*60)[0]), int(divmod(divmod(x, 60*60)[1], 60)[0])))
visit['start_time_human'] = pd.to_datetime(visit['start_time'], unit='s')
visit['end_time_human'] = pd.to_datetime(visit['end_time'], unit='s')

# Get all coordinates for map zoom display
visit_path_coordinates = sum(visit['path'].apply(lambda x: list(x.values() if isinstance(x, dict) else [])),[])
visit_coordinates = visit[['lat', 'lon']].apply(lambda x: [float(x['lat']), float(x['lon'])], axis=1).to_list()
activity_path_coordinates = sum(activity['path'].apply(lambda x: list(x.values() if isinstance(x, dict) else [])),[])
activity_start_coordinates = activity[['start_lat', 'start_lon']].apply(lambda x: [float(x['start_lat']), float(x['start_lon'])], axis=1).to_list()
activity_end_coordinates = activity[['end_lat', 'end_lon']].apply(lambda x: [float(x['end_lat']), float(x['end_lon'])], axis=1).to_list()
coordinates = sum([visit_path_coordinates, visit_coordinates, activity_path_coordinates, activity_start_coordinates, activity_end_coordinates], [])
min_lat = min((coord[0] for coord in coordinates if coord[0] is not None), default=None)
max_lat = max((coord[0] for coord in coordinates if coord[0] is not None), default=None)
min_lon = min((coord[1] for coord in coordinates if coord[1] is not None), default=None)
max_lon = max((coord[1] for coord in coordinates if coord[1] is not None), default=None)

pd.DataFrame(
      np.concatenate([
      activity[['start_time_human', 'end_time_human', 'start_time', 'end_time', 'start_location_id', 'start_location_name', 'end_location_id', 'end_location_name', 'vehicle_type']].to_numpy(), 
      visit[['start_time_human', 'end_time_human', 'start_time', 'end_time', 'location_id', 'location_name']].assign(end_location_id=None, end_location_name=None, vehicle_type=None).to_numpy()
      ])
      , columns=['start_time_human', 'end_time_human', 'start_time', 'end_time', 'start_location_id', 'start_location_name', 'end_location_id', 'end_location_name', 'vehicle_type']
      ).sort_values('start_time')


,start_time_human,end_time_human,start_time,end_time,start_location_id,start_location_name,end_location_id,end_location_name,vehicle_type
0,2025-02-28 08:09:42,2025-02-28 08:16:13,1740730182,1740730573,1,Home,5,807 Giải Phóng - giáp Kim Đồng,walking
1,2025-02-28 08:16:14,2025-02-28 08:18:01,1740730574,1740730681,5,807 Giải Phóng - giáp Kim Đồng,5,807 Giải Phóng - giáp Kim Đồng,in bus
2,2025-02-28 08:18:02,2025-02-28 08:47:11,1740730682,1740732431,5,807 Giải Phóng - giáp Kim Đồng,6,162 Khuất Duy Tiến - giáp Lê Văn Lương,in bus
5,2025-02-28 08:47:12,2025-02-28 18:22:18,1740732432,1740766938,2,Work,None,None,None
3,2025-02-28 18:22:19,2025-02-28 18:29:45,1740766939,1740767385,2,Work,7,39 Khuất Duy Tiến - giáp Tổ Hữu,walking
4,2025-02-28 18:29:46,2025-02-28 19:12:21,1740767386,1740769941,7,39 Khuất Duy Tiến - giáp Tổ Hữu,22,Đối Diện 807 Giải Phóng - giáp Định Công,in bus
6,2025-02-28 19:12:22,2025-02-28 19:18:26,1740769942,1740770306,1,Home,None,None,None
7,2025-02-28 19:18:27,2025-03-01 09:05:58,1740770307,1740819958,1,Home,None,None,None


In [10]:
import folium
from folium.plugins import MarkerCluster, AntPath

f = folium.Figure(width=1000, height=600)
m = folium.Map(
                location=[(min_lat + max_lat)/2, (min_lon + max_lon)/2],
                zoom_start=13, 
                control_scale=True,
                tiles="cartodbpositron",
               ).add_to(f)

# # if the points are too close to each other, cluster them, create a cluster overlay with MarkerCluster
marker_cluster = MarkerCluster().add_to(m)

for _, item in activity.iterrows():
    # print(item['path'].values())
    tooltip_txt = '<p>'\
                +'From: '\
                +pd.to_datetime(item['start_time'], unit = 's').strftime('%Y-%m-%d %H:%M:%S')\
                +'<br> To: '\
                +pd.to_datetime(item['end_time'], unit = 's').strftime('%Y-%m-%d %H:%M:%S')\
                +'<br> Transportation: '\
                +item['vehicle_type']\
                +'</p>'
    if isinstance(item['path'], dict):
        AntPath(
            locations=list(item['path'].values()), 
            color=item['color'],
            delay=800,
            weight=5,
            opacity=0.5,
            # reverse="True", 
            dash_array=[10, 20],
            tooltip=tooltip_txt
        ).add_to(m)

        for key, value in item['path'].items():
            folium.CircleMarker(
                value,
                radius=5,
                fill=True,
                color=None,
                fill_color = 'blue',
                fill_opacity=0.3,
                tooltip=pd.to_datetime(key, unit = 's').strftime('%Y-%m-%d %H:%M:%S')
            ).add_to(m)

    folium.Marker(
                    location = (item['start_lat'], item['start_lon']),
                    icon = folium.Icon(icon='play', prefix='glyphicon', color='orange'),
                    tooltip=tooltip_txt
                ).add_to(marker_cluster)    
    
    folium.Marker(
                    location = (item['end_lat'], item['end_lon']),
                    icon = folium.Icon(icon='stop', prefix='glyphicon', color='blue'),
                    tooltip=tooltip_txt
                ).add_to(marker_cluster)    
        
for _, item in visit.iterrows():
    if isinstance(item['path'], dict):
        folium.PolyLine(
            list(item['path'].values()), 
            color='black',
            weight=0.5,
            opacity=0.5,
            # dash_array='5, 5'
            ).add_to(m)
    
    tooltip_txt = '<p>'\
                +'From: '\
                +pd.to_datetime(item['start_time'], unit = 's').strftime('%Y-%m-%d %H:%M:%S')\
                +'<br> To: '\
                +pd.to_datetime(item['end_time'], unit = 's').strftime('%Y-%m-%d %H:%M:%S')\
                +'<br> Duration: '\
                +item['duration']\
                +'</p>'
    
    folium.Marker(
                    location = (item['lat'], item['lon']),
                    icon = folium.Icon(icon='ok', prefix='glyphicon', color='green'),
                    tooltip = tooltip_txt
                ).add_to(marker_cluster)    
    
m.fit_bounds(m.get_bounds())

m